<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c06-header.png" alt="Nextia Learning · Deep Learning and Transformers Explained" width="100%">

# Solution: Learning rate and optimizers

**[Learning rate and optimizers](https://learning.nextia-ai.com/courses/deep-learning/m04/learning-rate-and-optimizers/)** · Deep Learning and Transformers Explained · Module 4, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** read a training curve and say whether the learning rate is too small, too large or about right, and to compare SGD, momentum and Adam on Larkfield's ticket router.

| | |
|---|---|
| **Time** | About 30 minutes. The training runs take about 1 minute in total on the course's test computer. |
| **Needs** | An internet connection for the setup cells. No account. PyTorch (preinstalled in Colab and Kaggle). |
| **You should know** | Gradient descent and the learning rate, from [Gradient descent](https://learning.nextia-ai.com/courses/deep-learning/m03/gradient-descent/). The training loop and `train()`, from [The training loop](https://learning.nextia-ai.com/courses/deep-learning/m03/the-training-loop/). Training and validation curves, from [Batches, epochs and validation](https://learning.nextia-ai.com/courses/deep-learning/m03/batches-epochs-and-validation/). |
| **You do not need** | A graphics card or a local project. The setup below downloads the tickets and the course's code. |
| **Tested** | Python 3.12 with PyTorch 2.14.1 (CPU), in Jupyter on macOS, from a fresh start (*Restart and run all*); Colab and Kaggle have PyTorch preinstalled |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m04/learning-rate-and-optimizers/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C06/M04-L01-learning-rate-and-optimizers/learning-rate-and-optimizers-solution.ipynb).

## Setup: check PyTorch

Run the next cell. It uses the PyTorch that is already installed (Colab and Kaggle have it) or installs the CPU version if it is missing; that can take two minutes the first time. You should see a line that starts with `PyTorch 2.` and the device `cpu`. The course uses the CPU only, so a graphics card is not needed. 

In [ ]:
try:
    import torch
except ImportError:  # not installed: install the CPU version
    %pip install -q --disable-pip-version-check torch --index-url https://download.pytorch.org/whl/cpu
    import torch
import platform
torch.manual_seed(0)
print("PyTorch", torch.__version__, "| Python", platform.python_version(), "| device cpu")

## Setup: get the tickets and the course's code

Run the next cell. It downloads Larkfield's ticket texts (about 2 MB) and the course's `ticketnet.py` into a folder `ticket-router`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 4696 tickets, valid 600, test 600.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `ticket-router` (in Colab: the **Files** panel on the left), then run the cell again.

The tickets are made up for this course by a language model, from a list of situations; no real customer wrote them.

In [ ]:
# Setup: download the tickets and ticketnet.py into ticket-router/ and check them.
import hashlib, os, subprocess, sys, urllib.request
from pathlib import Path

LABS = os.environ.get("NX_LABS_RAW", "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C06/")
FILES = {
    "train.csv": "98487169346f8b0d9bc55a3844849ab95d012b75f01d7083605a6a1cea57aa39",
    "valid.csv": "f0c7bfa75e75837cdaf891ac8d007a78a018ba8e3488155cf43f2c6a34a064c8",
    "test.csv": "f373f402bcc9742cdcf8829720710a97b7a1dce5d8e34aaa88d03e7e49a5404e",
    "split_manifest.json": "d043cf720697054fb16e6cfff72ecb49b471d75038308ad5836779a0dd0bf8a5",
}
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-router").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:  # Python without certificates (python.org on macOS): try curl
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-router and run again.")
download(LABS + "project/ticketnet.py", PROJECT / "ticketnet.py")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
import ticketnet
from ticketnet import *  # TEAMS, Vocab, tokenize, the three networks, train(), evaluate() ...
train_rows, valid_rows, test_rows = (load_split(f"data/{p}.csv") for p in ("train", "valid", "test"))
print(f"Ready: train {len(train_rows)} tickets, valid {len(valid_rows)}, test {len(test_rows)}.")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `show_shape(name, tensor)` prints a tensor's shape, as the lessons do.

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

torch.set_num_threads(2)  # the numbers in the lessons were measured on a CPU

def fingerprint_tensor(t):
    return fingerprint([round(float(v), 4) for v in t.flatten().tolist()])

def show_shape(name, t):
    print(f"{name:<12} shape {tuple(t.shape)}")

## Setup: a plot helper

The next cell defines `plot_losses()`, which draws the training loss (dashed) and the validation loss (solid) of one or more runs. Run it. It prints nothing.

In [ ]:
# Helper for this lesson: plot the training and validation loss of one or more runs.
import matplotlib.pyplot as plt

def plot_losses(runs, title, top=None):
    """runs: {name: history}, each history a list of rows from train()."""
    fig, ax = plt.subplots(figsize=(7, 3.5))
    for name, history in runs.items():
        epochs = [row["epoch"] for row in history]
        line, = ax.plot(epochs, [row["valid_loss"] for row in history], marker="o", ms=3, label=f"{name}: validation")
        ax.plot(epochs, [row["train_loss"] for row in history], ls="--", color=line.get_color(), label=f"{name}: training")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    if top is not None:
        ax.set_ylim(0, top)
    ax.set_title(title)
    ax.legend(fontsize=8)
    plt.show()

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The problem: the same network, five different results

Priya trains the bag-of-words router of Module 3 five times. Only the **learning rate** changes: the tickets, the network, the starting weights (seed 0) and the 15 epochs stay the same. She uses plain gradient descent on mini-batches, the optimizer `"sgd"` of `train()`.

First, turn the tickets into bag-of-words tensors, as in Module 3. Run the cell.

In [ ]:
import torch
from torch import nn
from ticketnet import *       # the course's code: Vocab, bag_of_words, BagOfWordsNet, train() ...

train_rows, valid_rows = load_split("data/train.csv"), load_split("data/valid.csv")
vocab = Vocab([r["text"] for r in train_rows])          # words from the training tickets only
x_train, x_valid = bag_of_words(train_rows, vocab), bag_of_words(valid_rows, vocab)
y_train, y_valid = labels(train_rows), labels(valid_rows)
print("x_train", tuple(x_train.shape), " x_valid", tuple(x_valid.shape))

> **Check.** You should see `x_train (4696, 2864)  x_valid (600, 2864)`.

> **Predict.** Five learning rates: 0.00001, 0.001, 0.1, 1 and 10. Which one will give the best router after 15 epochs? Will any of them end **worse** than it started? Write your guess, then run the cell. It takes about 20 seconds.

In [ ]:
runs = {}
for learning_rate in [0.00001, 0.001, 0.1, 1, 10]:
    set_seed(0)                                      # the same starting weights for every run
    model = BagOfWordsNet(len(vocab))
    runs[learning_rate] = train(model, x_train, y_train, x_valid, y_valid, epochs=15,
                                lr=learning_rate, optimizer="sgd", log=None)
    first, last = runs[learning_rate][0], runs[learning_rate][-1]
    print(f"lr {learning_rate:<6g} train loss {first['train_loss']:.3f} -> {last['train_loss']:.3f}"
          f"   valid accuracy {last['valid_acc']:.1%}")

> **Check.** You should see:
>
> `lr 1e-05  train loss 1.624 -> 1.623   valid accuracy 18.3%`  
> `lr 0.001  train loss 1.619 -> 1.569   valid accuracy 39.7%`  
> `lr 0.1    train loss 1.188 -> 0.097   valid accuracy 98.0%`  
> `lr 1      train loss 1.051 -> 0.024   valid accuracy 97.8%`  
> `lr 10     train loss 1.579 -> 2.509   valid accuracy 27.5%`  

No line printed an error, but the results go from 18.3% to 98.0% right. At learning rate 10, the loss ends **higher** than it started, and 27.5% is exactly the share of `delivery` tickets in the validation set. A wrong learning rate is a silent failure.

## 2. The idea on a tiny example

One weight `w` and the loss `w × w`, whose slope is `2 × w`. Start at `w = 3` and take four steps with four learning rates. Run the cell.

In [ ]:
def descend(learning_rate, w=3.0, steps=4):
    """Walk downhill on the loss w * w, starting at w = 3."""
    path = [w]
    for _ in range(steps):
        slope = 2 * w                     # the slope of w * w at this w
        w = w - learning_rate * slope     # one step: learning rate x slope
        path.append(round(w, 3))
    return path

for learning_rate in [0.1, 0.4, 0.9, 1.1]:
    print(f"learning rate {learning_rate}: w = {descend(learning_rate)}")

> **Check.** You should see:
>
> `learning rate 0.1: w = [3.0, 2.4, 1.92, 1.536, 1.229]`  
> `learning rate 0.4: w = [3.0, 0.6, 0.12, 0.024, 0.005]`  
> `learning rate 0.9: w = [3.0, -2.4, 1.92, -1.536, 1.229]`  
> `learning rate 1.1: w = [3.0, -3.6, 4.32, -5.184, 6.221]`  

0.1 creeps towards 0 (too small), 0.4 gets there fast (good), 0.9 swings across 0 (unstable), and 1.1 swings further out every step: it **diverges**. Each step multiplies `w` by 1 − 2 × learning rate.

> **Your turn.** Compute the path for learning rate 0.5 with `descend()`, and store it in `path_05`. Then run the check cell.

In [ ]:
path_05 = descend(0.5)
print(path_05)

In [ ]:
expect_hash('path_05', path_05, 'b6888660e4205a94')

## 3. Read the curves: three shapes

The next cell draws three of the five runs. Look for the three shapes of the lesson: **too small** (both losses high and flat), **good** (fast, then slow), **unstable** (the loss jumps up again). Losses above 2.5 are cut at the top. Run the cell.

In [ ]:
plot_losses({"lr 0.001": runs[0.001], "lr 0.1": runs[0.1], "lr 1": runs[1]},
            "SGD with three learning rates", top=2.5)

> **Check.** You should see the two curves of learning rate 0.001 stay near 1.6, those of 0.1 fall smoothly, and those of 1 fall fast with jumps.

Now look at the unstable run, epoch by epoch. Run the cell.

In [ ]:
for row in runs[1][:8]:                               # SGD with learning rate 1, first 8 epochs
    print(f"epoch {row['epoch']}: train loss {row['train_loss']:.3f}, valid accuracy {row['valid_acc']:.1%}")

> **Check.** You should see:
>
> `epoch 1: train loss 1.051, valid accuracy 50.8%`  
> `epoch 2: train loss 0.363, valid accuracy 88.2%`  
> `epoch 3: train loss 0.160, valid accuracy 96.2%`  
> `epoch 4: train loss 0.323, valid accuracy 85.5%`  
> `epoch 5: train loss 0.085, valid accuracy 97.7%`  
> `epoch 6: train loss 0.236, valid accuracy 88.8%`  
> `epoch 7: train loss 1.433, valid accuracy 28.3%`  
> `epoch 8: train loss 0.216, valid accuracy 94.2%`  

> **Your turn.** Find the epoch where the run with learning rate 1 has its **highest** training loss, from `runs[1]`, and store the epoch number in `worst_epoch`. Hint: `max(..., key=lambda row: row["train_loss"])`. Then run the check cell.

In [ ]:
worst_epoch = max(runs[1], key=lambda row: row["train_loss"])["epoch"]
print(worst_epoch)

In [ ]:
expect('worst_epoch', worst_epoch, 7)

If you had stopped this run at that epoch, you would have saved a broken router. A run that jumps like this is luck, not a result: lower the learning rate.

## 4. Optimizers: three ways to take a step

The **optimizer** turns the gradients into weight changes (step 5 of the training loop). **SGD** takes the slope of each mini-batch. **Momentum** keeps a velocity, like a rolling ball. Run the cell: it compares them on a slope that stays at 1.

In [ ]:
learning_rate, slope = 0.1, 1.0       # a long, gentle slope: the same slope at every step
velocity, sgd_total, momentum_total = 0.0, 0.0, 0.0
for step in range(1, 5):
    velocity = 0.9 * velocity + slope           # momentum keeps 90% of the last push
    sgd_total += learning_rate * slope
    momentum_total += learning_rate * velocity
    print(f"step {step}: SGD moved {sgd_total:.3f}, momentum moved {momentum_total:.3f}")

> **Check.** You should see:
>
> `step 1: SGD moved 0.100, momentum moved 0.100`  
> `step 2: SGD moved 0.200, momentum moved 0.290`  
> `step 3: SGD moved 0.300, momentum moved 0.561`  
> `step 4: SGD moved 0.400, momentum moved 0.905`  

**Adam** gives each weight its own step size. The next cell takes one step of SGD and one of Adam on two weights: one with a steep slope (10) and one with a flat slope (0.01). Run it.

In [ ]:
for name in ["SGD", "Adam"]:
    w = torch.tensor([1.0, 1.0], requires_grad=True)       # two weights, both start at 1
    optimizer = torch.optim.SGD([w], lr=0.01) if name == "SGD" else torch.optim.Adam([w], lr=0.01)
    loss = 10 * w[0] + 0.01 * w[1]      # w[0] has a steep slope (10), w[1] a flat one (0.01)
    loss.backward()
    optimizer.step()
    steps = (1 - w.detach()).tolist()
    print(f"{name:<4} step of w[0]: {steps[0]:.5f}   step of w[1]: {steps[1]:.5f}")

> **Check.** You should see:
>
> `SGD  step of w[0]: 0.10000   step of w[1]: 0.00010`  
> `Adam step of w[0]: 0.01000   step of w[1]: 0.01000`  

SGD moves the steep weight 1,000 times more than the flat one. Adam moves both by about the learning rate.

> **Predict.** At learning rate 0.001 and at 0.1, which optimizer is fastest, and which is best after 15 epochs? Then run the cell. It takes about 25 seconds.

In [ ]:
for optimizer in ["sgd", "momentum", "adam"]:
    for learning_rate in [0.001, 0.1]:
        set_seed(0)
        model = BagOfWordsNet(len(vocab))
        history = train(model, x_train, y_train, x_valid, y_valid, epochs=15,
                        lr=learning_rate, optimizer=optimizer, log=None)
        print(f"{optimizer:<8} lr {learning_rate:<5g} valid accuracy after 1 epoch {history[0]['valid_acc']:.1%},"
              f" after 15 {history[-1]['valid_acc']:.1%}")

> **Check.** You should see:
>
> `sgd      lr 0.001 valid accuracy after 1 epoch 20.7%, after 15 39.7%`  
> `sgd      lr 0.1   valid accuracy after 1 epoch 77.3%, after 15 98.0%`  
> `momentum lr 0.001 valid accuracy after 1 epoch 29.0%, after 15 87.7%`  
> `momentum lr 0.1   valid accuracy after 1 epoch 92.7%, after 15 97.5%`  
> `adam     lr 0.001 valid accuracy after 1 epoch 97.2%, after 15 97.7%`  
> `adam     lr 0.1   valid accuracy after 1 epoch 96.2%, after 15 94.5%`  

The same learning rate means a different step size for each optimizer. Each one has its own good range, and their best results are close (98.0%, 97.5%, 97.7%).

## 5. Change one thing: momentum at learning rate 1

SGD with learning rate 1 is unstable but ends at 97.8%.

> **Predict.** Momentum makes steps up to 10 times longer on a steady slope. What happens to momentum at learning rate 1?

> **Your turn.** Train `BagOfWordsNet` with `optimizer="momentum"` and `lr=1` for 15 epochs (with `set_seed(0)` first, as above). Store the validation accuracy of the last epoch in `momentum_1_acc`. Then run the check cell.

In [ ]:
set_seed(0)
model = BagOfWordsNet(len(vocab))
history = train(model, x_train, y_train, x_valid, y_valid, epochs=15, lr=1, optimizer="momentum", log=None)
momentum_1_acc = history[-1]["valid_acc"]
print(momentum_1_acc)

In [ ]:
expect('momentum_1_acc', momentum_1_acc, 0.3167, tolerance=0.002)

Momentum at 1 diverges: the steps are far too long. For momentum, 0.1 is good.

## 6. The default, and how to check it

Start with **Adam at 0.001**. Check it with one run 10 times smaller and one 10 times larger, and compare the lowest validation loss and its epoch. Run the cell. It takes about 10 seconds.

In [ ]:
for learning_rate in [0.0001, 0.001, 0.01]:      # the default, 10 times smaller, 10 times larger
    set_seed(0)
    model = BagOfWordsNet(len(vocab))
    history = train(model, x_train, y_train, x_valid, y_valid, epochs=15, lr=learning_rate, log=None)
    best = min(history, key=lambda row: row["valid_loss"])
    print(f"Adam lr {learning_rate:<6g} best valid loss {best['valid_loss']:.3f} at epoch {best['epoch']:>2},"
          f" accuracy then {best['valid_acc']:.1%}")

> **Check.** You should see:
>
> `Adam lr 0.0001 best valid loss 0.286 at epoch 15, accuracy then 98.5%`  
> `Adam lr 0.001  best valid loss 0.063 at epoch 12, accuracy then 98.3%`  
> `Adam lr 0.01   best valid loss 0.058 at epoch  2, accuracy then 98.8%`  

At 0.0001, the best epoch is the last one: still too slow. At 0.01, the best epoch is the second one: very fast, then overfitting. The difference between 0.001 and 0.01 is 3 of 600 tickets, not a real difference, so keep the default.

## 7. Failure case: a learning rate carried over to another optimizer

Tomás switches Priya's script from SGD to Adam and keeps her learning rate of 0.1. Run the cell to see both Adam runs side by side.

In [ ]:
adam_runs = {}
for learning_rate in [0.1, 0.001]:
    set_seed(0)
    model = BagOfWordsNet(len(vocab))
    adam_runs[f"Adam lr {learning_rate}"] = train(model, x_train, y_train, x_valid, y_valid, epochs=15,
                                                   lr=learning_rate, log=None)
plot_losses(adam_runs, "Adam at SGD's learning rate (0.1) and at its own default (0.001)", top=0.6)
print({name: history[-1]["valid_acc"] for name, history in adam_runs.items()})

> **Check.** You should see `{'Adam lr 0.1': 0.945, 'Adam lr 0.001': 0.9767}`. At 0.1, the validation loss climbs from 0.104 to 0.443 with jumps; at 0.001, it falls smoothly.

A learning rate belongs to an optimizer. When you change the optimizer, start again from its default, and check 10 times smaller and larger.

## Recap

- The learning rate sets the step size. Too small: both losses high and flat. Too large: jumps, or divergence. Good: fast, then slow.
- SGD uses the slope of each batch; momentum keeps a velocity; Adam gives each weight its own step size.
- Start with Adam at 0.001, then check 10 times smaller and 10 times larger.

The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m04/learning-rate-and-optimizers/) has the learning-rate explorer with all 15 recorded runs, and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Overfitting controls](https://learning.nextia-ai.com/courses/deep-learning/m04/overfitting-controls/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/deep-learning/m04/learning-rate-and-optimizers/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.